# Lab 6.2: Double Dueling DQN on Atari Breakout
This notebook demonstrates a Double Dueling Deep Q-Network (DQN) implementation using `gymnasium` on the `Breakout` Atari environment.

## ✅ Step 1: Install Dependencies

In [1]:
#!pip install gymnasium[atari,accept-rom-license] opencv-python matplotlib

## ✅ Step 2: Import Libraries

In [2]:
import numpy as np
import random
import gymnasium as gym
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from collections import deque
import cv2
import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'tensorflow'

## ✅ Step 3: Frame Preprocessing and Stacking

In [ ]:
def preprocess_frame(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
    resized = cv2.resize(gray, (84, 84))
    normalized = resized / 255.0
    return normalized

class FrameStack:
    def __init__(self, stack_size=4):
        self.stack_size = stack_size
        self.frames = deque(maxlen=stack_size)

    def reset(self, frame):
        processed = preprocess_frame(frame)
        self.frames = deque([processed] * self.stack_size, maxlen=self.stack_size)
        return np.stack(self.frames, axis=-1)

    def step(self, frame):
        processed = preprocess_frame(frame)
        self.frames.append(processed)
        return np.stack(self.frames, axis=-1)

## ✅ Step 4: Build Double Dueling DQN Model

In [ ]:
def build_dueling_dqn(input_shape, num_actions):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(32, (8, 8), strides=4, activation='relu')(inputs)
    x = layers.Conv2D(64, (4, 4), strides=2, activation='relu')(x)
    x = layers.Conv2D(64, (3, 3), strides=1, activation='relu')(x)
    x = layers.Flatten()(x)

    # Dueling streams
    value = layers.Dense(512, activation='relu')(x)
    value = layers.Dense(1)(value)

    advantage = layers.Dense(512, activation='relu')(x)
    advantage = layers.Dense(num_actions)(advantage)

    q_values = value + (advantage - tf.reduce_mean(advantage, axis=1, keepdims=True))

    model = models.Model(inputs=inputs, outputs=q_values)
    model.compile(optimizer=optimizers.Adam(learning_rate=0.00025), loss='huber')
    return model

## ✅ Step 5: Initialize Environment

In [ ]:
env = gym.make('ALE/Breakout-v5')
n_actions = env.action_space.n
frame_stack = FrameStack()

main_model = build_dueling_dqn((84, 84, 4), n_actions)
target_model = build_dueling_dqn((84, 84, 4), n_actions)
target_model.set_weights(main_model.get_weights())

memory = deque(maxlen=100_000)

def remember(s, a, r, s_, done):
    memory.append((s, a, r, s_, done))

def act(state, epsilon):
    if np.random.rand() <= epsilon:
        return random.randrange(n_actions)
    q_values = main_model.predict(state[np.newaxis], verbose=0)
    return np.argmax(q_values[0])

def replay(batch_size, gamma):
    minibatch = random.sample(memory, batch_size)
    for state, action, reward, next_state, done in minibatch:
        target = main_model.predict(state[np.newaxis], verbose=0)
        next_q_main = main_model.predict(next_state[np.newaxis], verbose=0)
        next_q_target = target_model.predict(next_state[np.newaxis], verbose=0)
        best_action = np.argmax(next_q_main[0])
        target[0][action] = reward if done else reward + gamma * next_q_target[0][best_action]
        main_model.fit(state[np.newaxis], target, epochs=1, verbose=0)

## ✅ Step 6: Training Loop

In [ ]:
episodes = 100
batch_size = 32
gamma = 0.99
epsilon = 1.0
epsilon_min = 0.1
epsilon_decay = 0.995
scores = []

for e in range(episodes):
    obs, _ = env.reset()
    state = frame_stack.reset(obs)
    total_reward = 0
    done = False

    while not done:
        action = act(state, epsilon)
        obs_, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        next_state = frame_stack.step(obs_)
        remember(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward

        if len(memory) > batch_size:
            replay(batch_size, gamma)

    if epsilon > epsilon_min:
        epsilon *= epsilon_decay

    target_model.set_weights(main_model.get_weights())
    scores.append(total_reward)
    print(f"Episode {e+1}, Score: {total_reward}, Epsilon: {epsilon:.3f}")

## ✅ Step 7: Plot Results

In [ ]:
plt.plot(scores)
plt.title("Double Dueling DQN - Breakout")
plt.xlabel("Episode")
plt.ylabel("Total Reward")
plt.grid(True)
plt.show()
env.close()